# Step 4: GMFRMNet Dataset Loader & Architecture Verification (with Persistent Cache)

This cell sets up:
1. **Train/Val Split Caching (`train_val_split.json`)**: Automatically cached directly into `/content/drive/MyDrive/HRSCD_cache/` so splits remain 100% consistent across restarts.
2. **PyTorch Dataset & Augmentations**: Flips, rotations, normalization on $256 \times 256$ patches.
3. **GMFRMNet Complete Model Definition**: 2D Selective Scan (2DSS), VMamba Backbone, GLCSB Feature Fusion, MCSRB Decoder.
4. **Adaptive BCE + Dice Loss Function**.
5. **Model Smoke Test**: Runs a forward pass on GPU to verify tensor shapes before training.

In [ ]:
import os
import json
import glob
import random
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

PATCHES_DIR = "/content/dataset_patches"
# Persistent Google Drive cache directory
DRIVE_CACHE_DIR = "/content/drive/MyDrive/HRSCD_cache"
LOCAL_CACHE_DIR = "/content/cache"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active compute device: {device}")

# =====================================================================
# 1. PERSISTENT DATASET SPLIT CACHING (85% Train / 15% Validation)
# =====================================================================
SPLIT_CACHE_PATH = os.path.join(DRIVE_CACHE_DIR, "train_val_split.json")
LOCAL_SPLIT_CACHE = os.path.join(LOCAL_CACHE_DIR, "train_val_split.json")

lbl_files = sorted([os.path.basename(f) for f in glob.glob(f"{PATCHES_DIR}/train/label/*.png")])
print(f"Total patches detected: {len(lbl_files)}")

if os.path.exists(SPLIT_CACHE_PATH):
    with open(SPLIT_CACHE_PATH, "r") as f:
        split_data = json.load(f)
    train_files = split_data["train"]
    val_files = split_data["val"]
    print(f"[Cache Loaded] Restored dataset split: {len(train_files)} Train, {len(val_files)} Validation.")
else:
    random.seed(42)
    shuffled = lbl_files.copy()
    random.shuffle(shuffled)
    split_idx = int(0.85 * len(shuffled))
    train_files = sorted(shuffled[:split_idx])
    val_files = sorted(shuffled[split_idx:])
    split_data = {"train": train_files, "val": val_files}
    for p in [SPLIT_CACHE_PATH, LOCAL_SPLIT_CACHE]:
        try:
            with open(p, "w") as f:
                json.dump(split_data, f)
        except Exception:
            pass
    print(f"[Cache Created] Saved dataset split to Google Drive: {len(train_files)} Train, {len(val_files)} Validation.")

# =====================================================================
# 2. PYTORCH DATASET WITH AUGMENTATION
# =====================================================================
class ChangeDetectionDataset(Dataset):
    def __init__(self, root_dir, file_list, img_size=256, is_train=True):
        self.t1_dir = os.path.join(root_dir, "train", "t1")
        self.t2_dir = os.path.join(root_dir, "train", "t2")
        self.lbl_dir = os.path.join(root_dir, "train", "label")
        self.files = file_list
        self.is_train = is_train
        self.img_size = img_size
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        name = self.files[idx]
        t1 = Image.open(os.path.join(self.t1_dir, name)).convert("RGB")
        t2 = Image.open(os.path.join(self.t2_dir, name)).convert("RGB")
        lbl = Image.open(os.path.join(self.lbl_dir, name)).convert("L")
        
        # Standardize size
        if t1.size != (self.img_size, self.img_size):
            t1 = t1.resize((self.img_size, self.img_size), Image.BILINEAR)
            t2 = t2.resize((self.img_size, self.img_size), Image.BILINEAR)
            lbl = lbl.resize((self.img_size, self.img_size), Image.NEAREST)
            
        # Synchronized Data Augmentation (Train only)
        if self.is_train:
            if random.random() > 0.5:
                t1 = t1.transpose(Image.FLIP_LEFT_RIGHT)
                t2 = t2.transpose(Image.FLIP_LEFT_RIGHT)
                lbl = lbl.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                t1 = t1.transpose(Image.FLIP_TOP_BOTTOM)
                t2 = t2.transpose(Image.FLIP_TOP_BOTTOM)
                lbl = lbl.transpose(Image.FLIP_TOP_BOTTOM)
            if random.random() > 0.7:
                rot = random.choice([Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270])
                t1 = t1.transpose(rot)
                t2 = t2.transpose(rot)
                lbl = lbl.transpose(rot)
                
        t1_tensor = self.normalize(T.ToTensor()(t1))
        t2_tensor = self.normalize(T.ToTensor()(t2))
        lbl_tensor = torch.from_numpy((np.array(lbl) > 127).astype(np.float32)).unsqueeze(0)
        return t1_tensor, t2_tensor, lbl_tensor

train_dataset = ChangeDetectionDataset(PATCHES_DIR, train_files, is_train=True)
val_dataset = ChangeDetectionDataset(PATCHES_DIR, val_files, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f"DataLoaders ready: {len(train_loader)} train batches/epoch, {len(val_loader)} val batches/epoch.")

# =====================================================================
# 3. GMFRMNet ARCHITECTURE DEFINITION (2DSS + VMamba + GLCSB + MCSRB)
# =====================================================================
class SelectiveScan2D(nn.Module):
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.proj_in = nn.Conv2d(d_model, d_model, kernel_size=1)
        self.dw_conv = nn.Conv2d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)
        self.A = nn.Parameter(torch.randn(4, d_model, d_state) * 0.02)
        self.B_proj = nn.Linear(d_model, d_state)
        self.V_proj = nn.Linear(d_state, d_model)
        self.proj_out = nn.Conv2d(d_model, d_model, kernel_size=1)
        
    def forward(self, x):
        B, C, H, W = x.shape
        res = x
        u = self.dw_conv(self.proj_in(x))
        s1 = u.flatten(2).transpose(1, 2)
        s2 = torch.flip(u, dims=[2, 3]).flatten(2).transpose(1, 2)
        s3 = u.transpose(2, 3).flatten(2).transpose(1, 2)
        s4 = torch.flip(u.transpose(2, 3), dims=[2, 3]).flatten(2).transpose(1, 2)
        
        scans = [s1, s2, s3, s4]
        outputs = []
        for d, s in enumerate(scans):
            gate = torch.sigmoid(self.B_proj(s))
            decay = torch.sigmoid(self.A[d].mean(dim=0))
            h = gate * decay
            out_d = self.V_proj(h)
            outputs.append(out_d.transpose(1, 2).reshape(B, C, H, W))
            
        merged = (outputs[0] + torch.flip(outputs[1], dims=[2, 3]) + 
                  outputs[2].transpose(2, 3) + torch.flip(outputs[3], dims=[2, 3]).transpose(2, 3)) / 4.0
        return self.proj_out(merged) + res

class VSSBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.norm = nn.BatchNorm2d(in_channels)
        self.ssm = SelectiveScan2D(in_channels)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * 2, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1)
        )
        self.norm2 = nn.BatchNorm2d(in_channels)
        
    def forward(self, x):
        x = x + self.ssm(self.norm(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VMambaEncoder(nn.Module):
    def __init__(self, in_chans=3, dims=[64, 128, 256, 512]):
        super().__init__()
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_chans, dims[0] // 2, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(dims[0] // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(dims[0] // 2, dims[0], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(dims[0]),
            nn.ReLU(inplace=True)
        )
        self.stage1 = VSSBlock(dims[0])
        self.down1 = nn.Conv2d(dims[0], dims[1], kernel_size=3, stride=2, padding=1)
        self.stage2 = VSSBlock(dims[1])
        self.down2 = nn.Conv2d(dims[1], dims[2], kernel_size=3, stride=2, padding=1)
        self.stage3 = VSSBlock(dims[2])
        self.down3 = nn.Conv2d(dims[2], dims[3], kernel_size=3, stride=2, padding=1)
        self.stage4 = VSSBlock(dims[3])
        
    def forward(self, x):
        x = self.patch_embed(x)
        f1 = self.stage1(x)
        f2 = self.stage2(self.down1(f1))
        f3 = self.stage3(self.down2(f2))
        f4 = self.stage4(self.down3(f3))
        return [f1, f2, f3, f4]

class GLCSB(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        concat_channels = in_channels * 2
        self.phi_q = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        self.phi_k = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        self.phi_v = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        self.proj_d = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        self.ssm_local = SelectiveScan2D(concat_channels)
        self.conv_local = nn.Conv2d(concat_channels, in_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.fuse_conv = nn.Conv2d(in_channels * 2, in_channels, kernel_size=1)
        
    def forward(self, f_t1, f_t2):
        D = torch.cat([f_t1, f_t2], dim=1)
        B, _, H, W = D.shape
        Q = self.phi_q(D).flatten(2).transpose(1, 2)
        K = self.phi_k(D).flatten(2)
        V = self.phi_v(D).flatten(2).transpose(1, 2)
        scale = 1.0 / np.sqrt(Q.size(-1))
        attn = torch.softmax(torch.bmm(Q, K) * scale, dim=-1)
        F_global = torch.bmm(attn, V).transpose(1, 2).reshape(B, -1, H, W) + self.proj_d(D)
        F_local = self.relu(self.conv_local(self.ssm_local(D)))
        return self.fuse_conv(torch.cat([F_global, F_local], dim=1))

class MCSRB(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.conv3x3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.conv5x5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2)
        self.conv7x7 = nn.Conv2d(in_channels, in_channels, kernel_size=7, padding=3)
        fused_ch = in_channels * 4
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.channel_mlp = nn.Sequential(
            nn.Conv2d(fused_ch, fused_ch // 4, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(fused_ch // 4, fused_ch, kernel_size=1),
            nn.Sigmoid()
        )
        self.spatial_conv = nn.Sequential(nn.Conv2d(2, 1, kernel_size=1), nn.Sigmoid())
        self.out_conv = nn.Sequential(
            nn.Conv2d(fused_ch, in_channels, kernel_size=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, F_in):
        S1, S2, S3, S4 = self.conv1x1(F_in), self.conv3x3(F_in), self.conv5x5(F_in), self.conv7x7(F_in)
        J = torch.cat([S1, S2, S3, S4], dim=1)
        J_C = self.channel_mlp(self.gap(J)) * J
        J_max, _ = torch.max(J, dim=1, keepdim=True)
        J_avg = torch.mean(J, dim=1, keepdim=True)
        J_S = self.spatial_conv(torch.cat([J_max, J_avg], dim=1)) * J
        return self.out_conv(J + J_C + J_S)

class GMFRMNet(nn.Module):
    def __init__(self, in_chans=3, num_classes=1, dims=[64, 128, 256, 512]):
        super().__init__()
        self.encoder = VMambaEncoder(in_chans=in_chans, dims=dims)
        self.glcsb4 = GLCSB(dims[3])
        self.glcsb3 = GLCSB(dims[2])
        self.glcsb2 = GLCSB(dims[1])
        self.glcsb1 = GLCSB(dims[0])
        
        self.mcsrb1 = MCSRB(dims[3])
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.proj3 = nn.Conv2d(dims[3] + dims[2], dims[2], kernel_size=1)
        
        self.mcsrb2 = MCSRB(dims[2])
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.proj2 = nn.Conv2d(dims[2] + dims[1], dims[1], kernel_size=1)
        
        self.mcsrb3 = MCSRB(dims[1])
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.proj1 = nn.Conv2d(dims[1] + dims[0], dims[0], kernel_size=1)
        
        self.mcsrb4 = MCSRB(dims[0])
        self.pred_head = nn.Sequential(
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=True),
            nn.Conv2d(dims[0], dims[0] // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(dims[0] // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(dims[0] // 2, num_classes, kernel_size=1)
        )
        
    def forward(self, t1, t2):
        f1_t1, f2_t1, f3_t1, f4_t1 = self.encoder(t1)
        f1_t2, f2_t2, f3_t2, f4_t2 = self.encoder(t2)
        
        F4 = self.glcsb4(f4_t1, f4_t2)
        F3 = self.glcsb3(f3_t1, f3_t2)
        F2 = self.glcsb2(f2_t1, f2_t2)
        F1 = self.glcsb1(f1_t1, f1_t2)
        
        X1 = self.mcsrb1(F4)
        X2 = self.mcsrb2(self.proj3(torch.cat([self.up1(X1), F3], dim=1)))
        X3 = self.mcsrb3(self.proj2(torch.cat([self.up2(X2), F2], dim=1)))
        X4 = self.mcsrb4(self.proj1(torch.cat([self.up3(X3), F1], dim=1)))
        return self.pred_head(X4)

# Verification Forward Pass
test_model = GMFRMNet().to(device)
d_t1 = torch.randn(2, 3, 256, 256).to(device)
d_t2 = torch.randn(2, 3, 256, 256).to(device)
with torch.no_grad():
    out = test_model(d_t1, d_t2)
print(f"\n✓ GMFRMNet instantiated successfully! Output tensor: {out.shape} on {device}")


# Step 5: Training Loop with Auto-Checkpointing & Disruption Recovery

### 🛡️ Crash & Disruption Recovery Mechanism:
- **Live Cache Path**: Saves `checkpoint_latest.pth` after **every single epoch** directly into Google Drive (`/content/drive/MyDrive/HRSCD_cache/`).
- **Best Weights**: Automatically preserves `best_gmfrmnet.pth` whenever validation F1 score improves.
- **Seamless Resume**: If Colab disconnects, crashes, or times out, re-running this cell detects the cache file and resumes directly from the exact epoch where it was interrupted, preserving the model weights, optimizer, and scheduler states.

In [ ]:
import time
from tqdm import tqdm

# =====================================================================
# ADAPTIVE LOSS FUNCTION (Eq. 14-17 in Paper)
# =====================================================================
class AdaptiveBCEWithDiceLoss(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.bce = nn.BCEWithLogitsLoss(reduction="mean")
        
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        l_bce = self.bce(logits, targets)
        intersection = torch.sum(probs * targets)
        union = torch.sum(probs) + torch.sum(targets)
        l_dice = 1.0 - (2.0 * intersection + self.eps) / (union + self.eps)
        denom = l_bce.detach() + l_dice.detach() + self.eps
        alpha = l_bce.detach() / denom
        beta = l_dice.detach() / denom
        return alpha * l_bce + beta * l_dice, l_bce.item(), l_dice.item()

def calculate_metrics(preds, targets, threshold=0.5):
    preds_bin = (preds > threshold).astype(np.uint8).flatten()
    targets_bin = (targets > threshold).astype(np.uint8).flatten()
    tp = np.sum((preds_bin == 1) & (targets_bin == 1))
    fp = np.sum((preds_bin == 1) & (targets_bin == 0))
    fn = np.sum((preds_bin == 0) & (targets_bin == 1))
    tn = np.sum((preds_bin == 0) & (targets_bin == 0))
    eps = 1e-7
    oa = (tp + tn) / (tp + fp + fn + tn + eps)
    pre = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    f1 = 2 * pre * rec / (pre + rec + eps)
    iou = tp / (tp + fp + fn + eps)
    return oa * 100, iou * 100, f1 * 100, pre * 100, rec * 100

# =====================================================================
# TRAINING SETUP & AUTO-RESUME INITIALIZATION
# =====================================================================
TOTAL_EPOCHS = 60
model = GMFRMNet(in_chans=3, num_classes=1).to(device)
criterion = AdaptiveBCEWithDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()

CHECKPOINT_CACHE = os.path.join(DRIVE_CACHE_DIR, "checkpoint_latest.pth")
BEST_MODEL_PATH = os.path.join(DRIVE_CACHE_DIR, "best_gmfrmnet.pth")
LOCAL_CHECKPOINT = os.path.join(LOCAL_CACHE_DIR, "checkpoint_latest.pth")

start_epoch = 1
best_val_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_iou": []}

# Check for existing checkpoint cache
resume_path = None
if os.path.exists(CHECKPOINT_CACHE):
    resume_path = CHECKPOINT_CACHE
elif os.path.exists(LOCAL_CHECKPOINT):
    resume_path = LOCAL_CHECKPOINT

if resume_path:
    print(f"\n[Cache Found] Loading checkpoint from: {resume_path}...")
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_f1 = ckpt.get("best_val_f1", 0.0)
    history = ckpt.get("history", history)
    print(f"[Restored] Successfully resumed from Epoch {start_epoch - 1} | Best Val F1: {best_val_f1:.2f}%")
else:
    print("\n[Fresh Start] No checkpoint cache detected. Starting training from Epoch 1.")

# =====================================================================
# MAIN TRAINING LOOP (WITH AUTOMATIC PER-EPOCH CACHING)
# =====================================================================
for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    train_loss_total = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} [Train]")
    for t1, t2, lbl in pbar:
        t1, t2, lbl = t1.to(device), t2.to(device), lbl.to(device)
        optimizer.zero_grad()
        
        # Mixed Precision forward pass
        with torch.cuda.amp.autocast():
            preds = model(t1, t2)
            loss, _, _ = criterion(preds, lbl)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss_total += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    scheduler.step()
    train_avg_loss = train_loss_total / len(train_loader)
    
    # Evaluation Phase
    model.eval()
    val_loss_total = 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for t1, t2, lbl in tqdm(val_loader, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} [Val]"):
            t1, t2, lbl = t1.to(device), t2.to(device), lbl.to(device)
            with torch.cuda.amp.autocast():
                preds = model(t1, t2)
                loss, _, _ = criterion(preds, lbl)
            val_loss_total += loss.item()
            probs = torch.sigmoid(preds).cpu().numpy()
            all_preds.append(probs)
            all_targets.append(lbl.cpu().numpy())
            
    val_avg_loss = val_loss_total / len(val_loader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    oa, iou, f1, pre, rec = calculate_metrics(all_preds, all_targets)
    
    elapsed = time.time() - epoch_start
    print(f"\nEpoch {epoch:02d}/{TOTAL_EPOCHS:02d} [{elapsed:.1f}s] - Train Loss: {train_avg_loss:.4f} | Val Loss: {val_avg_loss:.4f} | IoU: {iou:.2f}% | F1: {f1:.2f}% | OA: {oa:.2f}%")
    
    # Record history
    history["train_loss"].append(train_avg_loss)
    history["val_loss"].append(val_avg_loss)
    history["val_f1"].append(f1)
    history["val_iou"].append(iou)
    
    # Build checkpoint state dictionary
    state = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_val_f1": max(best_val_f1, f1),
        "history": history,
    }
    
    # 🛡️ SAVE TO RESUME CACHE (Drive + Local)
    for p in [CHECKPOINT_CACHE, LOCAL_CHECKPOINT]:
        try:
            torch.save(state, p)
        except Exception:
            pass
            
    # Save best model
    if f1 > best_val_f1:
        best_val_f1 = f1
        for p in [BEST_MODEL_PATH, "/content/best_gmfrmnet.pth"]:
            try:
                torch.save(model.state_dict(), p)
            except Exception:
                pass
        print(f"  ★ New Best Model Saved! F1 Score: {best_val_f1:.2f}%")
        
print("\n=======================================================")
print(f"Training Finished! Best Validation F1: {best_val_f1:.2f}%")
print(f"Best weights saved to: {BEST_MODEL_PATH}")
print("=======================================================")
